# Preliminary data processing

This script takes in the results from assembles-datasets and creates certain userful variables in the data.

In [1]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib as mpl
import matplotlib.ticker as ticker

import numpy as np
import pandas as pd
from pydmd import DMD
import os
import plotly.graph_objects as go
import plotly.express as px
import pickle

# import clustering packages
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import seaborn as sns
from celluloid import Camera

plt.style.use('custom.mplstyle')
%config InlineBackend.figure_format = 'retina'
from tqdm import tqdm

from stoch_sim_model import *

## 0. Load data and build datasets

In [2]:
# Load data from infections
reg_model = ''
runs = '-1-'
comment = "sparse-reg" #"Nact-Ediv-vir" #"prim-Nact-Ediv-vir" #"full-reg-vir" #"act-reg-exp-reg" # mem-reg # comp_bias-Nact-Ediv-vir

d_mean = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/stacked_data'+runs+'runs'+'-'+comment+'.pkl'
mean_df = pd.read_pickle(d_mean)

with pd.option_context('display.max_columns', None):
    display(mean_df)

,psi_Nact_I,psi_Nact_H,psi_Nact_IH,F0_Nact,psi_NM_I,psi_NM_H,psi_NM_IH,F0_NM,psi_EM_I,psi_EM_H,psi_EM_IH,F0_EM,psi_Ediv_I,psi_Ediv_H,psi_Ediv_IH,F0_Ediv,d_I,K_IE,b_I,S_0,I_0,d_S,d_IE,d_IH,K_IH,b_H,d_H,K_EI,K_EH,N_0,max_Na,b_myc,d_myc,myc_thresh,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_act,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_pM,T_pM_min,int_pE,int_pH,min_pS
0,-2.0,-2.0,-2.0,-2.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,-2.0,0.25,10000.000000,7.500000e-08,10000000.0,1000.0,0.01,16.0,0.0,50000.0,1.0,2.0,10000.000000,50000.0,200.0,4.0,4.0,0.4,1.0,0.5,1.0,0.25,0.333333,0.5,2.5,0.25,1.801637e+07,20.1390,0.000,4.496086e+06,0.000000,0.0,0.0000,21.000000,0.0,0.000000,0.0000,2.059752e+06,2.781867e+06
1,-2.0,-2.0,-2.0,-2.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,-2.0,0.25,10000.000000,9.583333e-08,10000000.0,1000.0,0.01,16.0,0.0,50000.0,1.0,2.0,10000.000000,50000.0,200.0,4.0,4.0,0.4,1.0,0.5,1.0,0.25,0.333333,0.5,2.5,0.25,3.305401e+07,14.7945,0.000,8.258839e+06,0.000000,0.0,0.0000,21.000000,0.0,0.000000,0.0000,4.010199e+06,7.344823e+05
2,-2.0,-2.0,-2.0,-2.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,-2.0,0.25,10000.000000,1.166667e-07,10000000.0,1000.0,0.01,16.0,0.0,50000.0,1.0,2.0,10000.000000,50000.0,200.0,4.0,4.0,0.4,1.0,0.5,1.0,0.25,0.333333,0.5,2.5,0.25,3.815177e+07,11.7495,0.000,9.535319e+06,0.000000,0.0,0.0000,21.000000,0.0,0.000000,0.0000,4.700033e+06,4.629696e+05
3,-2.0,-2.0,-2.0,-2.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,-2.0,0.25,10000.000000,1.375000e-07,10000000.0,1000.0,0.01,16.0,0.0,50000.0,1.0,2.0,10000.000000,50000.0,200.0,4.0,4.0,0.4,1.0,0.5,1.0,0.25,0.333333,0.5,2.5,0.25,4.021801e+07,9.7650,0.000,1.005270e+07,0.000000,0.0,0.0000,21.000000,0.0,0.000000,0.0000,4.980202e+06,3.253319e+05
4,-2.0,-2.0,-2.0,-2.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,-2.0,0.25,10000.000000,1.583333e-07,10000000.0,1000.0,0.01,16.0,0.0,50000.0,1.0,2.0,10000.000000,50000.0,200.0,4.0,4.0,0.4,1.0,0.5,1.0,0.25,0.333333,0.5,2.5,0.25,4.127224e+07,8.3790,0.000,1.031665e+07,0.000000,0.0,0.0000,21.000000,0.0,0.000000,0.0000,5.122245e+06,2.423731e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89149370,2.0,2.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,1.00,464158.883361,2.000000e-07,10000000.0,1000.0,0.01,16.0,0.0,50000.0,1.0,2.0,464158.883361,50000.0,200.0,4.0,4.0,0.4,1.0,0.5,1.0,0.25,0.333333,0.5,2.5,0.25,6.035950e+06,9.0510,13.041,7.081409e+06,237534.328625,144648.0,13.2405,7.001768,525037.0,28.432078,480259.0380,3.540705e+06,3.110130e+06
89149371,2.0,2.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,1.00,1000000.000000,1.375000e-07,10000000.0,1000.0,0.01,16.0,0.0,50000.0,1.0,2.0,1000000.000000,50000.0,200.0,4.0,4.0,0.4,1.0,0.5,1.0,0.25,0.333333,0.5,2.5,0.25,2.713874e+06,20.3070,0.000,2.709445e+06,0.000000,0.0,0.0000,21.000000,0.0,0.000000,0.0000,1.250952e+06,7.002154e+06
89149372,2.0,2.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,1.00,1000000.000000,1.583333e-07,10000000.0,1000.0,0.01,16.0,0.0,50000.0,1.0,2.0,1000000.000000,50000.0,200.0,4.0,4.0,0.4,1.0,0.5,1.0,0.25,0.333333,0.5,2.5,0.25,6.212996e+06,14.2695,0.000,6.212243e+06,109.048081,77.0,15.5295,0.533977,44.0,0.000000,135.7335,3.060266e+06,4.087514e+06
89149373,2.0,2.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,1.00,1000000.000000,1.791667e-07,10000000.0,1000.0,0.01,16.0,0.0,50000.0,1.0,2.0,1000000.000000,50000.0,200.0,4.0,4.0,0.4,1.0,0.5,1.0,0.25,0.333333,0.5,2.5,0.25,6.560564e+06,11.0670,0.000,7.006965e+06,81401.792636,47554.0,15.5190,4.667197,139525.0,5.728828,148095.0240,3.501413e+06,3.395662e+06


## 1. Understanding the statistics of responses to an infection

In [3]:
# Create additional variables
virs = np.unique(mean_df[['I_0','d_I','K_IE','b_I','N_0']].to_numpy(), axis = 0)

module_reg = {'Nact': Nact_reg, 'NM': NM_reg, 'EM': EM_reg, 'Ediv': Ediv_reg}
module_reg_labels = {'Nact': param_names[-16:-12], 'NM': param_names[-12:-8], 'EM': param_names[-8:-4], 'Ediv': param_names[-4:]}

if "full" in comment:
    reg = Nact_reg + NM_reg + EM_reg + Ediv_reg
    reg_label = param_names[-len(reg):]
    modules = ['Nact', 'NM', 'EM', 'Ediv']
    reg_and = [Nact_reg[2]] + [NM_reg[2]] + [EM_reg[2]] + [Ediv_reg[2]]
    reg_bias = [Nact_reg[3]] + [NM_reg[3]] + [EM_reg[3]] + [Ediv_reg[3]]
else:
    reg = (Nact_reg if "Nact" in comment else []) + (NM_reg if "NM" in comment else []) + (EM_reg if "EM" in comment else []) + (Ediv_reg if "Ediv" in comment else [])
    reg_label = (param_names[-16:-12] if "Nact" in comment else []) + (param_names[-12:-8] if "NM" in comment else []) + (param_names[-8:-4] if "EM" in comment else []) + (param_names[-4:] if "Ediv" in comment else [])
    modules = (["Nact"] if "Nact" in comment else []) + (["NM"] if "NM" in comment else []) + (["EM"] if "EM" in comment else []) + (["Ediv"] if "Ediv" in comment else [])
    reg_or = ([Nact_reg[0:2]] if "Nact" in comment else []) + ([NM_reg[0:2]] if "NM" in comment else []) + ([EM_reg[0:2]] if "EM" in comment else []) + ([Ediv_reg[0:2]] if "Ediv" in comment else [])
    reg_and = ([Nact_reg[2]] if "Nact" in comment else []) + ([NM_reg[2]] if "NM" in comment else []) + ([EM_reg[2]] if "EM" in comment else []) + ([Ediv_reg[2]] if "Ediv" in comment else [])
    reg_and_label = ([param_names[-14]] if "Nact" in comment else []) + ([param_names[-10]] if "NM" in comment else []) + ([param_names[-6]] if "EM" in comment else []) + ([param_names[-2]] if "Ediv" in comment else [])
    reg_bias = ([Nact_reg[3]] if "Nact" in comment else []) + ([NM_reg[3]] if "NM" in comment else []) + ([EM_reg[3]] if "EM" in comment else []) + ([Ediv_reg[3]] if "Ediv" in comment else [])
    reg_bias_label = ([param_names[-13]] if "Nact" in comment else []) + ([param_names[-9]] if "NM" in comment else []) + ([param_names[-5]] if "EM" in comment else []) + ([param_names[-1]] if "Ediv" in comment else [])

# Build variables

mean_df['antigenicity_over_harm'] = antigenicity_over_harm(mean_df)
mean_df['T_pE_clear'] = mean_df['T_pE_max'] - mean_df['T_max_pI']
mean_df['max_pM_fold'] = mean_df['max_pM']/mean_df['N_0']

mean_df['int_pE_fold'] = mean_df['int_pE']/mean_df['N_0']

# identify Biologically evidenced networks
keep_vars = ['harm_pI','T_min_pI', 'harm_pS', "max_pM_fold", "T_pM_min", "T_max_pI", "T_pE_start", "T_pE_max", "int_pE_fold", 'min_pS', 'antigenicity_over_harm', 'T_pE_clear'] 
#keep_vars = ['harm_pI','T_min_pI', 'harm_pS', "max_pM_fold", "T_max_pI", "T_pE_start", "T_pE_max", "int_pE_fold", 'min_pS', 'antigenicity_over_harm', 'T_pE_clear']

In [ ]:
# save data sets
cutoff = 1 - np.minimum((1 + psi_max)**len(modules), 1000)/(mean_df.shape[0]/len(virs))
infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_IE, b_I, N_0) in enumerate(virs):
    data = mean_df.loc[(mean_df["d_I"] == d_I)*(mean_df["K_IE"] == K_IE)*(mean_df["b_I"] == b_I), ['b_I','d_I', 'K_IE', 'I_0','S_0', 'N_0', 'd_S', 'K_EH'] + Nact_reg + NM_reg + EM_reg + Ediv_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_IE = K_IE, d_I = d_I, b_I = b_I)
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/(b_S*sim_duration)
    data.loc[:,"peff_protection"] = (no_eff_stats[3] - data['harm_pI'].to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))
    data.loc[:,"peff_utility"] = (no_eff_stats[3] + 1.0*no_eff_stats[4] - (data['harm_pI'] + data['harm_pS']).to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))

    if 'comp_model' not in comment:
        data.loc[:, "high_putility"] = 1*(data['peff_utility'] >= np.quantile(data['peff_utility'], cutoff))
        data.loc[:, "high_pprotection"] = 1*(data['peff_protection'] >= np.quantile(data['peff_protection'], cutoff))
        data.loc[:, "high_ptoxicity"] = 1*(data['peff_toxicity'] >= np.quantile(data['peff_toxicity'], cutoff))
        data.loc[:, "high_pmemory"] = 1*(data['max_pM_fold'] >= np.quantile(data['max_pM_fold'], cutoff))
        data.loc[:, "high_pmemory_survive"] = 1*(data['T_pM_min'] >= np.quantile(data['T_pM_min'], cutoff))
        data.loc[:, "low_clear_timing"] = 1*(-data['T_max_pI'] >= np.quantile(-data['T_max_pI'], cutoff))
        data.loc[:, "low_resp_timing"] = 1*(-data['T_pE_start'] >= np.quantile(-data['T_pE_start'], cutoff))
        data.loc[:, "low_resp_clear_timing"] = 1*(-data['T_pE_clear'] >= np.quantile(-data['T_pE_clear'], cutoff))
        data.loc[:, "high_presponse"] = 1*(data['int_pE_fold'] >= np.quantile(data['int_pE_fold'], cutoff))

    infection_scenarios.append(data)

# stack datasets
clustered_mean_df = pd.concat(infection_scenarios)
clustered_mean_df.to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(infection_scenarios, f)

/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWar

In [ ]:
del clustered_mean_df, infection_scenarios